# Fine-tuning roberta on DepressionEmo

Multi-label classification of 8 emotions in Reddit posts (anger, cognitive dysfunction, emptiness, hopelessness, loneliness, sadness, suicide intent, worthlessness).

**How to run on Google Colab**
1. `Runtime → Change runtime type → GPU` (T4 is enough).
2. Run the cells top to bottom. When asked, upload `train.json`, `val.json` and `test.json`.
3. Tip: set `SMOKE_TEST = True` in the config cell for a 1-minute dry run first, then set it back to `False`.


## 1. Install and check the GPU

In [ ]:
!pip install -q transformers accelerate scikit-learn sentencepiece protobuf

In [ ]:
import torch, transformers
print("torch:", torch.__version__, "| transformers:", transformers.__version__)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("No GPU found. Training will be very slow. Use Runtime > Change runtime type > GPU.")

## 2. Configuration

In [ ]:
import sys, os
IN_COLAB = "google.colab" in sys.modules

# ---- paths ----
DATA_DIR   = "/content/data" if IN_COLAB else "../data"   # folder with train/val/test .json
OUTPUT_DIR = "roberta-depressionemo"

# ---- model / training ----
MODEL_NAME = "roberta-base"   # stable + strong for ~4k multi-label records (DeBERTa-v3 can go NaN)
MAX_LENGTH = 256
EPOCHS     = 6
BATCH_SIZE = 16      # lower to 8 if you get a CUDA out-of-memory error
LR         = 2e-5      # typical for roberta-base. If the loss turns NaN, the best checkpoint so far is restored
SEED       = 42
POS_WEIGHT = False   # True = upweight rare emotions in the loss
PRECISION  = "fp32"   # "fp32" = safest (default). "bf16" = Ampere+ GPUs (A100/L4). "fp16" = fast on T4 and safe with roberta-base
SMOKE_TEST = False   # True = tiny run (200 train / 100 val+test examples) to check the pipeline
SMOKE_EPOCHS = 1     # epochs used when SMOKE_TEST=True. Use ~15 for an "overfit test": training loss should fall far below 0.5

## 3. Get the data

On Colab the cell below asks you to upload the three files. If you prefer Google Drive, skip the upload and instead run:

```python
from google.colab import drive
drive.mount("/content/drive")
DATA_DIR = "/content/drive/MyDrive/path/to/data"   # folder containing train.json, val.json, test.json
```

In [ ]:
have_files = os.path.isdir(DATA_DIR) and len(os.listdir(DATA_DIR)) > 0
if not have_files:
    if IN_COLAB:
        from google.colab import files
        os.makedirs(DATA_DIR, exist_ok=True)
        print("Select train.json, val.json and test.json")
        uploaded = files.upload()
        for name, content in uploaded.items():
            with open(os.path.join(DATA_DIR, name), "wb") as f:
                f.write(content)
    else:
        raise FileNotFoundError(f"DATA_DIR '{DATA_DIR}' not found (current directory: {os.getcwd()})")
print("Files in DATA_DIR:", sorted(os.listdir(DATA_DIR)))

## 4. Data helpers

In [ ]:
import glob, json
import numpy as np

LABELS = [
    "anger", "cognitive dysfunction", "emptiness", "hopelessness",
    "loneliness", "sadness", "suicide intent", "worthlessness",
]

# Substring -> canonical label, in case the dataset spells a name differently
KEYWORDS = {
    "cog": "cognitive dysfunction", "forget": "cognitive dysfunction",
    "brain": "cognitive dysfunction", "suicid": "suicide intent",
    "anger": "anger", "empt": "emptiness", "hopeless": "hopelessness",
    "lonel": "loneliness", "sad": "sadness", "worthless": "worthlessness",
}


def normalize_emotion(name):
    n = str(name).strip().lower().replace("_", " ")
    if n in LABELS:
        return n
    for key, label in KEYWORDS.items():
        if key in n:
            return label
    return None


def read_records(path):
    with open(path, "r", encoding="utf-8") as f:
        raw = f.read().strip()
    try:
        obj = json.loads(raw)
    except json.JSONDecodeError:  # JSON Lines
        return [json.loads(line) for line in raw.splitlines() if line.strip()]
    if isinstance(obj, list):
        return obj
    if isinstance(obj, dict):
        if any(k in obj for k in ("text", "post", "emotions")):
            return [obj]                      # a single record
        lists = [v for v in obj.values() if isinstance(v, list)]
        if lists:
            return lists[0]                   # wrapper like {"data": [...]}
        values = list(obj.values())
        if values and all(isinstance(v, dict) for v in values):
            return values                     # {id: record, ...}
    raise ValueError(f"Unrecognised JSON structure in {path}")


def find_split_file(data_dir, names):
    if not os.path.isdir(data_dir):
        raise FileNotFoundError(f"data dir '{data_dir}' does not exist (current directory: {os.getcwd()})")
    hits = []
    for ext in ("json", "jsonl"):
        for name in names:
            hits += glob.glob(os.path.join(data_dir, "**", f"*{name}*.{ext}"), recursive=True)
    if not hits:
        raise FileNotFoundError(f"No file matching {names} under {data_dir}. Contents: {sorted(os.listdir(data_dir))}")
    return sorted(set(hits))[0]


def records_to_xy(records, split_name):
    """Labels are rebuilt from the emotion NAMES (never from a precomputed vector)."""
    texts, ys, unknown = [], [], set()
    for r in records:
        if "text" in r and r["text"]:
            text = r["text"]
        else:
            text = f'{r.get("title", "")} ### {r.get("post", "")}'
        emo = r.get("emotions")
        if emo is None:
            raise KeyError(f'[{split_name}] no "emotions" field. Keys found: {sorted(r)}')
        if isinstance(emo, str):
            emo = [e for e in emo.replace(";", ",").split(",") if e.strip()]
        y = [0.0] * len(LABELS)
        for e in emo:
            label = normalize_emotion(e)
            if label is None:
                unknown.add(e)
            else:
                y[LABELS.index(label)] = 1.0
        texts.append(str(text))
        ys.append(y)
    if unknown:
        raise ValueError(f"[{split_name}] unmapped emotion names: {unknown}. Add them to KEYWORDS.")
    return texts, np.array(ys, dtype=np.float32)

## 5. Load the data
Expected sizes: 4225 train / 906 val / 906 test. If your counts differ, check the printed file paths.

In [ ]:
splits = {}
for key, names, limit in [("train", ["train"], 200), ("val", ["val", "dev"], 100), ("test", ["test"], 100)]:
    path = find_split_file(DATA_DIR, names)
    texts, y = records_to_xy(read_records(path), key)
    if SMOKE_TEST:
        texts, y = texts[:limit], y[:limit]
    splits[key] = (texts, y)
    print(f"{key:5s}: {len(texts):5d} examples  <- {path}")

print()
print("Label prevalence in train:")
for lab, frac in zip(LABELS, splits["train"][1].mean(0)):
    print(f"  {lab:22s} {frac:.3f}")

## 6. Tokenizer, datasets and model

In [ ]:
import torch
from torch.utils.data import Dataset
from transformers import (AutoModelForSequenceClassification, AutoTokenizer,
                          DataCollatorWithPadding, Trainer, TrainingArguments, set_seed)

set_seed(SEED)
tok = AutoTokenizer.from_pretrained(MODEL_NAME)


class DS(Dataset):
    def __init__(self, texts, y):
        self.enc = tok(texts, truncation=True, max_length=MAX_LENGTH)
        self.y = y

    def __len__(self):
        return len(self.y)

    def __getitem__(self, i):
        item = {k: v[i] for k, v in self.enc.items()}
        item["labels"] = self.y[i].tolist()
        return item


ds = {k: DS(*v) for k, v in splits.items()}

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=len(LABELS),
    problem_type="multi_label_classification",
    id2label=dict(enumerate(LABELS)),
    label2id={l: i for i, l in enumerate(LABELS)},
)

# ---- tokenizer sanity check ----
ids = ds["train"].enc["input_ids"]
print("example tokens:", tok.convert_ids_to_tokens(ids[0])[:25])
unk_share = np.mean([t == tok.unk_token_id for seq in ids for t in seq])
trunc_share = np.mean([len(seq) >= MAX_LENGTH for seq in ids])
print(f"UNK token share: {unk_share:.4f} | share of posts cut at {MAX_LENGTH} tokens: {trunc_share:.1%}")
print("Healthy: readable word pieces like '▁I', '▁feel' and an UNK share close to 0.")

## 7. Metrics and trainer

In [ ]:
import inspect
from sklearn.metrics import f1_score


def sigmoid(x):
    return 1.0 / (1.0 + np.exp(-x))


def f1_scores(y_true, y_pred):
    return {
        "f1_macro": f1_score(y_true, y_pred, average="macro", zero_division=0),
        "f1_micro": f1_score(y_true, y_pred, average="micro", zero_division=0),
    }


def tune_thresholds(y_true, probs):
    """Best per-label decision threshold on the validation set."""
    grid = np.arange(0.10, 0.91, 0.05)
    best = []
    for j in range(y_true.shape[1]):
        scores = [f1_score(y_true[:, j], probs[:, j] >= t, zero_division=0) for t in grid]
        best.append(float(grid[int(np.argmax(scores))]))
    return np.array(best)


def compute_metrics(p):
    logits = p.predictions[0] if isinstance(p.predictions, tuple) else p.predictions
    return f1_scores(p.label_ids, sigmoid(logits) >= 0.5)


class MultiLabelTrainer(Trainer):
    pw = None  # optional per-label positive weights

    def compute_loss(self, model, inputs, return_outputs=False, **kw):
        labels = inputs.pop("labels")
        out = model(**inputs)
        pw = self.pw.to(out.logits.device) if self.pw is not None else None
        loss = torch.nn.functional.binary_cross_entropy_with_logits(
            out.logits, labels.float(), pos_weight=pw)
        if not torch.isfinite(loss):
            raise FloatingPointError(
                f"Loss became {loss.item()} at step {self.state.global_step}: training diverged. "
                "Use PRECISION='fp32' and/or a lower LR, then restart.")
        return (loss, out) if return_outputs else loss


if POS_WEIGHT:
    y_tr = splits["train"][1]
    MultiLabelTrainer.pw = torch.tensor(
        (len(y_tr) - y_tr.sum(0)) / np.maximum(y_tr.sum(0), 1), dtype=torch.float32)

# Argument names differ between transformers versions: pick what this install supports.
ta_params = inspect.signature(TrainingArguments.__init__).parameters
assert PRECISION in ("fp32", "bf16", "fp16"), "PRECISION must be 'fp32', 'bf16' or 'fp16'"
use_bf16 = PRECISION == "bf16" and torch.cuda.is_available()
use_fp16 = PRECISION == "fp16" and torch.cuda.is_available()
print(f"precision: {'bf16' if use_bf16 else 'fp16' if use_fp16 else 'fp32'} | lr: {LR}")

ta = dict(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=64,
    learning_rate=LR,
    num_train_epochs=SMOKE_EPOCHS if SMOKE_TEST else EPOCHS,
    weight_decay=0.01,
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    greater_is_better=True,
    save_total_limit=2,
    logging_steps=10 if SMOKE_TEST else 50,
    report_to="none",
    seed=SEED,
    bf16=use_bf16,
    fp16=use_fp16,
)
ta["eval_strategy" if "eval_strategy" in ta_params else "evaluation_strategy"] = "epoch"
if "warmup_ratio" in ta_params:
    ta["warmup_ratio"] = 0.1
else:
    ta["warmup_steps"] = 0.1   # newer versions accept a float ratio

tr_params = inspect.signature(Trainer.__init__).parameters
tok_kw = {"processing_class": tok} if "processing_class" in tr_params else {"tokenizer": tok}

trainer = MultiLabelTrainer(
    model=model, args=TrainingArguments(**ta),
    train_dataset=ds["train"], eval_dataset=ds["val"],
    data_collator=DataCollatorWithPadding(tok),
    compute_metrics=compute_metrics, **tok_kw)

## 8. Train
Validation F1 is printed after every epoch; the best checkpoint (by validation F1-macro) is restored at the end.

In [ ]:
try:
    trainer.train()
except FloatingPointError as e:
    # Loss became NaN/inf: keep the best checkpoint saved before the divergence.
    print("WARNING:", e)
    ckpt = trainer.state.best_model_checkpoint
    if ckpt is None:
        raise RuntimeError("Diverged before any checkpoint was saved. Lower LR (e.g. 5e-6) and keep PRECISION='fp32'.") from e
    print(f"Restoring best checkpoint so far: {ckpt} (val f1_macro = {trainer.state.best_metric})")
    _best = AutoModelForSequenceClassification.from_pretrained(ckpt)
    model.load_state_dict(_best.state_dict())
    del _best

## 9. Evaluate on the test set
Thresholds are tuned per emotion on the **validation** set, then applied unchanged to the test set.

In [ ]:
import pandas as pd


def logits_of(pred):
    p = pred.predictions
    return p[0] if isinstance(p, tuple) else p


val_logits  = logits_of(trainer.predict(ds["val"]))
test_logits = logits_of(trainer.predict(ds["test"]))
assert np.isfinite(val_logits).all() and np.isfinite(test_logits).all(), (
    "The model outputs NaN/inf: training diverged. Retrain with PRECISION='fp32' and a lower LR.")
val_p, test_p = sigmoid(val_logits), sigmoid(test_logits)
yv, yt = splits["val"][1], splits["test"][1]

thr = tune_thresholds(yv, val_p)

summary = pd.DataFrame({
    "threshold 0.5":         f1_scores(yt, test_p >= 0.5),
    "tuned per-label thr.":  f1_scores(yt, test_p >= thr),
}).T
print(summary.round(4), "\n")

per_label = pd.DataFrame({
    "emotion":   LABELS,
    "threshold": thr,
    "test F1":   [f1_score(yt[:, j], test_p[:, j] >= thr[j], zero_division=0) for j in range(len(LABELS))],
    "support":   yt.sum(0).astype(int),
}).set_index("emotion")
print(per_label.round(4))

## 10. Save the results and the model

In [ ]:
os.makedirs(OUTPUT_DIR, exist_ok=True)
results = {
    "test_threshold_0.5": f1_scores(yt, test_p >= 0.5),
    "test_tuned_thresholds": f1_scores(yt, test_p >= thr),
    "thresholds": dict(zip(LABELS, thr.tolist())),
    "per_label_f1_tuned": per_label["test F1"].to_dict(),
}
with open(os.path.join(OUTPUT_DIR, "results.json"), "w") as f:
    json.dump(results, f, indent=2)

best_dir = os.path.join(OUTPUT_DIR, "best")
trainer.save_model(best_dir)
tok.save_pretrained(best_dir)
print("Saved to", best_dir)

Colab's disk is wiped when the session ends. Zip and download the model, or copy it to Google Drive:

In [ ]:
import shutil
shutil.make_archive("deberta-depressionemo-best", "zip", best_dir)
if IN_COLAB:
    from google.colab import files
    files.download("deberta-depressionemo-best.zip")
    # Or copy to Drive instead:
    # from google.colab import drive; drive.mount("/content/drive")
    # shutil.copytree(best_dir, "/content/drive/MyDrive/deberta-depressionemo-best")

## 11. Try it on a new text
(Placeholder example. Replace it with your own text. Research use only.)

In [ ]:
def predict(text):
    model.eval()
    enc = tok(text, return_tensors="pt", truncation=True, max_length=MAX_LENGTH).to(model.device)
    with torch.no_grad():
        probs = torch.sigmoid(model(**enc).logits)[0].cpu().numpy()
    table = pd.DataFrame({"probability": probs, "threshold": thr, "predicted": probs >= thr}, index=LABELS)
    return table.round(3)

predict("I haven't talked to anyone in weeks and I feel like nothing I do matters anymore.")